In [0]:
%pip uninstall -y investsphere-platform

In [0]:
%pip install -e /Workspace/Users/engineersantoshnair@outlook.com/investsphere-demo

In [0]:
dbutils.library.restartPython()

In [0]:
import sys
sys.path.insert(0, '/Workspace/Users/engineersantoshnair@outlook.com/investsphere-demo/src')
import investsphere_platform

In [0]:
import investsphere_platform
print("Library imported successfully")

In [0]:
import sys
sys.path.append("/Workspace/Users/<your-email>/investsphere_demo/pipelines")

In [0]:
import pipelines

In [0]:
%sql
SHOW TABLES IN investsphere.bronze;

In [0]:
 path = "/Volumes/investsphere/bronze/raw/investment_asset_master"
 print("FILES")
 for f in dbutils.fs.ls(path):
     print(" ", f.name, f.size)

In [0]:
df = spark.read.format("csv").option("header", "true").option("inferSchema", "true").load("/Volumes/investsphere/bronze/raw/investment_asset_master")
print("ROW_COUNT:", df.count())
df.show()

In [0]:
dbutils.fs.rm("/Volumes/investsphere/bronze/_checkpoints", recurse=True)
dbutils.fs.rm("/Volumes/investsphere/bronze/_schemas", recurse=True)
print("cleared checkpoint + schema state -- now re-run the bronze ingest job")              

In [0]:
print("RAW ROOT:")
for f in dbutils.fs.ls("/Volumes/investsphere/bronze/raw/"):
    print(" ", f.name)

In [0]:
spark.sql("SHOW TABLES IN investsphere.bronze").show(truncate=False)

In [0]:
%sql
CREATE VOLUME IF NOT EXISTS investsphere.bronze.`_checkpoints`;

In [0]:
%sql
CREATE VOLUME IF NOT EXISTS investsphere.bronze.`_schemas`;

In [0]:
spark.sql("SHOW TABLES IN investsphere.bronze").show(truncate=False)
spark.table("investsphere.bronze.raw_investment_asset_master").count()

In [0]:
from pyspark.sql import functions as F
# clean slate so the test is repeatable
spark.sql("DROP TABLE IF EXISTS investsphere.bronze.raw_test_asset")
dbutils.fs.rm("/Volumes/investsphere/bronze/_checkpoints/test_asset", True)
dbutils.fs.rm("/Volumes/investsphere/bronze/_schemas/test_asset", True)

df = (spark.readStream.format("cloudFiles")
      .option("cloudFiles.format", "csv")
      .option("cloudFiles.schemaLocation", "/Volumes/investsphere/bronze/_schemas/test_asset")
      .option("header", "true")
      .load("/Volumes/investsphere/bronze/raw/investment_asset_master"))

q = (df.writeStream.format("delta")
     .option("checkpointLocation", "/Volumes/investsphere/bronze/_checkpoints/test_asset")
     .trigger(availableNow=True)
     .toTable("investsphere.bronze.raw_test_asset")
     )

q.awaitTermination()
print("Rows written:", spark.table("investsphere.bronze.raw_test_asset").count())

In [0]:
spark.sql("DROP TABLE IF EXISTS investsphere.bronze.raw_test_asset")

In [0]:
dbutils.fs.rm("/Volumes/investsphere/bronze/_checkpoints/test_asset", True)
dbutils.fs.rm("/Volumes/investsphere/bronze/_schemas/test_asset",  True)

In [0]:
%sql
show tables in investsphere.bronze;

In [0]:
%sql
SELECT * from investsphere.bronze.raw_investment_transactions limit 10;

In [0]:
%sql
select transaction_id, _ingest_ts, _source_file, _batch_id from investsphere.bronze.raw_investment_transactions

In [0]:
%sql
select * 
from investsphere.silver.silver_transaction limit 10;

In [0]:
%sql
select * from investsphere.silver.quarantine_transaction;

In [0]:
%sql
SELECT * from investsphere.gold.fact_portfolio_exposure
order by portfolio_id, exposure_pct desc;

In [0]:
%sql
SELECT * FROM investsphere.gold.fact_limit_breach order by breach_by_pct desc;

In [0]:
%sql
show tables in investsphere.silver;

In [0]:
%sql
SELECT * from investsphere.governance.dq_results order by check_timestamp desc;

In [0]:
%sql
